# Databricks Lakeflow Jobs — YouTube Presentation Guide

> **Series:** DataMindAI | Ahmed | Head of Data Engineering  
> **Purpose:** Step-by-step notebook code for all 11 features in the presentation  
> **Data:** Dummy sales data (8 orders) — no external dependencies required

---
## How to use this notebook
- Run cells top to bottom on first setup (creates Bronze/Silver/Gold tables)
- Each feature section is self-contained — you can demo them independently
- All code uses `demo_catalog.bronze/silver/gold` — update catalog name if needed

## Pre-requisites
```
1. Databricks workspace (Free Edition or above — NOT Community Edition)
2. Unity Catalog enabled
3. Run the setup cell below once to create the catalog/schema structure
```

In [ ]:
# ── SETUP: Run once ──────────────────────────────────────────────────────
# Creates the catalog and schema structure used by all features.
# Update 'demo_catalog' to your own catalog name if needed.

spark.sql('CREATE CATALOG IF NOT EXISTS demo_catalog')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.gold')
print('Catalog and schemas ready.')

---
## FEATURE 01 — Job Anatomy: Ingest → Transform → Serve
---

## Feature 01 — Multi-Task Job Anatomy

Create these three notebooks as **separate files** in Databricks, then wire them
together as tasks in a single Lakeflow Job.

| Task | Notebook | Depends on |
|------|----------|------------|
| 1 | `01_ingest` | — |
| 2 | `02_transform` | ingest_bronze (ALL_SUCCESS) |
| 3 | `03_serve` | transform_silver (ALL_SUCCESS) |

### Task 1 — ingest_bronze

In [ ]:
# ── 01_ingest.py ─────────────────────────────────────────────────────────
# Creates dummy sales data and writes to Bronze Delta table

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

sales_data = [
    Row(order_id='ORD-001', product='Laptop',   region='North', revenue=1200.00, units=2, order_date='2024-01-15'),
    Row(order_id='ORD-002', product='Monitor',  region='South', revenue=450.00,  units=3, order_date='2024-01-15'),
    Row(order_id='ORD-003', product='Keyboard', region='East',  revenue=85.00,   units=5, order_date='2024-01-15'),
    Row(order_id='ORD-004', product='Laptop',   region='West',  revenue=1200.00, units=1, order_date='2024-01-15'),
    Row(order_id='ORD-005', product='Mouse',    region='North', revenue=35.00,   units=10, order_date='2024-01-15'),
    Row(order_id='ORD-006', product='Monitor',  region='East',  revenue=450.00,  units=2, order_date='2024-01-15'),
    Row(order_id='ORD-007', product='Headset',  region='South', revenue=150.00,  units=4, order_date='2024-01-15'),
    Row(order_id='ORD-008', product='Laptop',   region='East',  revenue=1200.00, units=3, order_date='2024-01-15'),
]

df = spark.createDataFrame(sales_data)
df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.raw_sales')

print(f'Ingested {df.count()} records to demo_catalog.bronze.raw_sales')
df.show()

### Task 2 — transform_silver

In [ ]:
# ── 02_transform.py ──────────────────────────────────────────────────────
from pyspark.sql.functions import col, current_timestamp, round as spark_round

df_bronze = spark.table('demo_catalog.bronze.raw_sales')

df_silver = (df_bronze
    .withColumn('line_total', spark_round(col('revenue') * col('units'), 2))
    .withColumn('load_ts', current_timestamp())
    .filter(col('revenue') > 0)
)

df_silver.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.silver.clean_sales')
print(f'Transformed {df_silver.count()} records')
df_silver.show()

### Task 3 — build_gold

In [ ]:
# ── 03_serve.py ──────────────────────────────────────────────────────────
from pyspark.sql.functions import sum as spark_sum, count, avg, round as spark_round

df_silver = spark.table('demo_catalog.silver.clean_sales')

df_gold = (df_silver
    .groupBy('region', 'product')
    .agg(
        spark_sum('line_total').alias('total_revenue'),
        count('order_id').alias('order_count'),
        spark_round(avg('units'), 1).alias('avg_units_per_order')
    )
    .orderBy('region', 'product')
)

df_gold.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.gold.revenue_by_region')
print('Gold table created:')
df_gold.show()

---
## FEATURE 04 — Task Dependencies (ALL_SUCCESS / ALL_FAILED)
---

## Feature 04 — Error Handler Task

Add this as a 4th task with **Run if: ALL_FAILED** on `ingest_bronze`.
To test: add `raise Exception('Simulated failure!')` to `01_ingest`, run, then remove it.

In [ ]:
# ── 04_error_notifier.py ─────────────────────────────────────────────────
# Task type: Notebook | Dependency condition: ALL_FAILED
import datetime

run_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('=' * 60)
print('PIPELINE FAILURE DETECTED')
print(f'Time: {run_time}')
print('Upstream tasks failed. On-call team notified.')
print('Check the Run History tab for detailed error logs.')
print('=' * 60)

from pyspark.sql import Row
df_audit = spark.createDataFrame([
    Row(event='pipeline_failure', detected_at=run_time, job='sales_pipeline')
])
df_audit.write.format('delta').mode('append').saveAsTable('demo_catalog.bronze.pipeline_audit')
print('Failure event written to pipeline_audit table.')

---
## FEATURE 05 — Dynamic Task Values (Revenue Passing)
---

## Feature 05 — Task Values: Revenue Routing

**Job setup:**
- Task 1: `05a_calculate_revenue` — publishes `total_revenue` via taskValues
- Task 2: `05b_revenue_routing` — reads it and routes to PREMIUM or STANDARD

**Task 2 parameter in the UI:**
```
Key: total_revenue  |  Value: {{tasks.calculate_revenue.values.total_revenue}}
```

In [ ]:
# ── 05a_calculate_revenue.py ────────────────────────────────────────────
# PUBLISHES: total_revenue, total_orders, avg_order_value
from pyspark.sql import Row
from pyspark.sql.functions import sum as spark_sum

sales = [
    Row(order_id='ORD-001', product='Laptop',  region='North', revenue=1200.00, units=2),
    Row(order_id='ORD-002', product='Monitor', region='South', revenue=450.00,  units=3),
    Row(order_id='ORD-003', product='Keyboard',region='East',  revenue=85.00,   units=5),
    Row(order_id='ORD-004', product='Laptop',  region='West',  revenue=1200.00, units=1),
    Row(order_id='ORD-005', product='Mouse',   region='North', revenue=35.00,   units=10),
]
df = spark.createDataFrame(sales)

total_revenue   = df.agg(spark_sum('revenue')).collect()[0][0]
total_orders    = df.count()
avg_order_value = round(total_revenue / total_orders, 2)

print(f'Total Revenue  : £{total_revenue:,.2f}')
print(f'Total Orders   : {total_orders}')
print(f'Avg Order Value: £{avg_order_value:,.2f}')

dbutils.jobs.taskValues.set(key='total_revenue',   value=float(total_revenue))
dbutils.jobs.taskValues.set(key='total_orders',    value=int(total_orders))
dbutils.jobs.taskValues.set(key='avg_order_value', value=float(avg_order_value))
print('Task values published.')

In [ ]:
# ── 05b_revenue_routing.py ──────────────────────────────────────────────
# CONSUMES total_revenue via widget parameter:
#   Key: total_revenue | Value: {{tasks.calculate_revenue.values.total_revenue}}

total_revenue = float(dbutils.widgets.get('total_revenue'))

print(f'Revenue received from upstream task: £{total_revenue:,.2f}')

THRESHOLD = 3000.00

if total_revenue >= THRESHOLD:
    print(f'HIGH REVENUE — £{total_revenue:,.2f} >= £{THRESHOLD:,.2f}')
    tier = 'PREMIUM'
else:
    print(f'STANDARD DAY — £{total_revenue:,.2f} < £{THRESHOLD:,.2f}')
    tier = 'STANDARD'

dbutils.jobs.taskValues.set(key='processing_tier', value=tier)
print(f'Processing tier set: {tier}')

---
## FEATURE 06 — If/Else Conditionals (Weekday vs Weekend)
---

## Feature 06 — Conditional Branching

**Job setup:**
- Add an **If/Else Condition** task: `route_by_day`
- Condition expression: `job.start_time.iso_weekday >= 6`
- TRUE branch → `06b_weekend_aggregation`
- FALSE branch → `06a_weekday_summary`

> To force the weekend branch on a weekday during demo, change condition to `>= 1`

In [ ]:
# ── 06a_weekday_summary.py ──────────────────────────────────────────────
# Runs on: FALSE branch (Mon–Fri)
import datetime

today = datetime.date.today()
print(f'WEEKDAY SUMMARY — {today.strftime("%A %d %b %Y")}')

from pyspark.sql import Row
from pyspark.sql.functions import sum as spark_sum

sales = [
    Row(region='North', revenue=1200.0), Row(region='South', revenue=450.0),
    Row(region='East',  revenue=750.0),  Row(region='West',  revenue=980.0),
]
df = spark.createDataFrame(sales)
df.groupBy('region').agg(spark_sum('revenue').alias('daily_revenue')).show()

In [ ]:
# ── 06b_weekend_aggregation.py ──────────────────────────────────────────
# Runs on: TRUE branch (Sat–Sun)
import datetime

today = datetime.date.today()
print(f'WEEKEND FULL AGGREGATION — {today.strftime("%A %d %b %Y")}')

from pyspark.sql import Row
from pyspark.sql.functions import sum as spark_sum, count

weekly = []
for day in range(7):
    for prod, price in [('Laptop',1200),('Monitor',450),('Keyboard',85)]:
        for region in ['North','South','East','West']:
            weekly.append(Row(day=day, product=prod, region=region, revenue=float(price)))

df = spark.createDataFrame(weekly)
(df.groupBy('product','region')
   .agg(spark_sum('revenue').alias('weekly_revenue'))
   .orderBy('weekly_revenue', ascending=False)
   .show(10))

---
## FEATURE 07 — For-Each Loops (5 Regions in Parallel)
---

## Feature 07 — For-Each Loops

**Job setup:**
- Task 1: `07a_generate_regions` — publishes `regions` list
- For-Each task: `process_each_region`
  - Inputs: `{{tasks.generate_regions.values.regions}}`
  - Concurrency: 3
  - Nested task: `07b_process_region`
    - Parameter: `input: {{input}}`

In [ ]:
# ── 07a_generate_regions.py ─────────────────────────────────────────────
# PUBLISHES regions list as a task value
import json

regions = [
    {'region_id': 'R001', 'region_name': 'North',   'target_table': 'north_sales',   'budget': 50000},
    {'region_id': 'R002', 'region_name': 'South',   'target_table': 'south_sales',   'budget': 45000},
    {'region_id': 'R003', 'region_name': 'East',    'target_table': 'east_sales',    'budget': 38000},
    {'region_id': 'R004', 'region_name': 'West',    'target_table': 'west_sales',    'budget': 42000},
    {'region_id': 'R005', 'region_name': 'Central', 'target_table': 'central_sales', 'budget': 60000},
]

for r in regions:
    print(f'  {r["region_id"]} — {r["region_name"]} (budget: £{r["budget"]:,})')

dbutils.jobs.taskValues.set(key='regions', value=json.dumps(regions))
print(f'Published {len(regions)} regions for parallel processing.')

In [ ]:
# ── 07b_process_region.py ───────────────────────────────────────────────
# INNER TASK — runs once per region in the For-Each loop
# Receives current iteration via widget parameter 'input'
import json, random

params      = json.loads(dbutils.widgets.get('input'))
region_id   = params['region_id']
region_name = params['region_name']
target_tbl  = params['target_table']
budget      = params['budget']

print(f'Processing: {region_name} ({region_id}) | Budget: £{budget:,}')

random.seed(hash(region_id) % 100)
from pyspark.sql import Row

data = [Row(region=region_name, product=p, revenue=float(pr), units=random.randint(1,10))
        for p, pr in [('Laptop',1200),('Monitor',450),('Keyboard',85),('Mouse',35),('Headset',150)]]
df = spark.createDataFrame(data)

total = df.agg({'revenue':'sum'}).collect()[0][0]
status = 'OVER' if total > budget else 'UNDER'
print(f'Revenue: £{total:,.2f} | Budget: £{budget:,} | {status} budget')

df.write.format('delta').mode('overwrite').option('overwriteSchema','true')
   .saveAsTable(f'demo_catalog.silver.{target_tbl}')

---
## FEATURE 08 — SQL Integration (Parameters In, Results Out)
---

## Feature 08 — SQL Integration

**Job setup:**
- Task 1: `08a_set_run_date` — publishes `run_date`
- Task 2: SQL task using `08_daily_revenue.sql`
  - Parameter in SQL: `{{tasks.set_run_date.values.run_date}}`
- Task 3: `08b_process_sql_results`
  - Parameter: `sql_results: {{tasks.daily_revenue_sql.output.rows}}`

Save the SQL below as a `.sql` file in your Databricks Repo.

In [ ]:
# ── 08a_set_run_date.py ─────────────────────────────────────────────────
# Run date published for the SQL task to consume
run_date = '2024-01-15'   # matches order_date in the dummy data

dbutils.jobs.taskValues.set(key='run_date', value=run_date)
print(f'run_date published: {run_date}')

### SQL file: 08_daily_revenue.sql
Save this as a `.sql` file in your Databricks Repo, then reference it in the SQL task.
```sql
-- 08_daily_revenue.sql
-- Parameter: {{tasks.set_run_date.values.run_date}}
-- Output rows available as: {{tasks.daily_revenue_sql.output.rows}}

SELECT
    region,
    COUNT(order_id)       AS order_count,
    SUM(revenue * units)  AS total_revenue,
    ROUND(AVG(revenue),2) AS avg_unit_price,
    SUM(units)            AS total_units
FROM demo_catalog.bronze.raw_sales
WHERE order_date = '{{tasks.set_run_date.values.run_date}}'
GROUP BY region
ORDER BY total_revenue DESC
```

In [ ]:
# ── 08b_process_sql_results.py ──────────────────────────────────────────
# Receives SQL output rows via widget parameter:
#   Key: sql_results | Value: {{tasks.daily_revenue_sql.output.rows}}
import json

rows = json.loads(dbutils.widgets.get('sql_results'))

print(f'Received {len(rows)} rows from SQL task')
print(f'{"Region":<12} {"Orders":>8} {"Revenue":>14} {"Avg Price":>12}')
print('-' * 52)

total_rev = 0
for row in rows:
    region   = row.get('region', 'Unknown')
    orders   = row.get('order_count', 0)
    revenue  = float(row.get('total_revenue', 0))
    avg_p    = float(row.get('avg_unit_price', 0))
    total_rev += revenue
    print(f'{region:<12} {orders:>8} £{revenue:>12,.2f} £{avg_p:>10,.2f}')

print('-' * 52)
print(f'{"TOTAL":<12} {"":>8} £{total_rev:>12,.2f}')
dbutils.jobs.taskValues.set(key='daily_total_revenue', value=float(total_rev))

---
## FEATURE 09 — Partial Runs & Partial Repairs
---

## Feature 09 — Partial Runs & Partial Repairs

**Partial Run (development):**
1. Job page → three-dot menu → Run with different parameters
2. In 'Tasks to run': deselect all, check only `transform_silver`
3. Only that task runs — others show as SKIPPED

**Partial Repair (production recovery):**
1. Add `raise Exception('Simulated failure!')` to `01_ingest` and run
2. Job fails on `ingest_bronze`
3. Remove the raise line
4. In the failed run: click **Repair Run** → only failed + downstream tasks re-run
5. Same run_id is preserved — no data duplication

> Key message: Repair picks up exactly where it left off. In a 10-task pipeline,
> this saves significant compute and avoids re-processing upstream data.

In [ ]:
# ── Simulate a failure for the Partial Repair demo ────────────────────
# Add this line to 01_ingest.py temporarily, then run the job.
# Remove it before running the Repair.

# raise Exception('Simulated production failure!')   # <-- uncomment to test
print('Normal execution — no failure simulated.')

---
## FEATURE 10 — Alerts & Duration Thresholds
---

## Feature 10 — Alerts & Notifications

**Configure in Job UI:**
- Job → Edit → Notifications → + Add notification
- Events: Job failure / Job success / Task failure
- Destinations: Email / Slack webhook URL / PagerDuty API URL

**Duration thresholds:**
- Expected completion: 30 seconds (for demo)
- Maximum completion: 5 minutes

Run `10_slow_job.py` below and uncomment `time.sleep(300)` to trigger the warning.

In [ ]:
# ── 10_slow_job.py ──────────────────────────────────────────────────────
import time

print('Job started — simulating normal processing...')

for i in range(5):
    print(f'Processing step {i+1}/5...')
    time.sleep(5)

# Uncomment to trigger Expected Completion alert:
# time.sleep(300)   # 5 minutes — will exceed 30-second threshold

print('Job completed successfully.')

---
## FEATURE 11 — Databricks Asset Bundles (DABs)
---

## Feature 11 — Asset Bundles (Jobs as Code)

Run these commands in your **local terminal**, not in Databricks.

```bash
# Install CLI
pip install databricks-cli

# Configure
databricks configure --token

# Validate
databricks bundle validate

# Deploy to dev
databricks bundle deploy -t dev

# Run deployed job
databricks bundle run sales_pipeline
```

The `databricks.yml` file below defines the full 3-task job as code.

### databricks.yml — Complete Job Definition
```yaml
bundle:
  name: lakeflow_demo_bundle

workspace:
  host: https://your-workspace.azuredatabricks.net

resources:
  jobs:
    sales_pipeline:
      name: Sales Pipeline (DAB Managed)
      schedule:
        quartz_cron_expression: '0 6 * * *'
        timezone_id: UTC
      email_notifications:
        on_failure:
          - your-email@domain.com
      job_clusters:
        - job_cluster_key: main_cluster
          new_cluster:
            spark_version: 15.4.x-scala2.12
            node_type_id: Standard_DS3_v2
            num_workers: 2
      tasks:
        - task_key: ingest_bronze
          notebook_task:
            notebook_path: ./notebooks/01_ingest
          job_cluster_key: main_cluster
        - task_key: transform_silver
          depends_on:
            - task_key: ingest_bronze
          notebook_task:
            notebook_path: ./notebooks/02_transform
          job_cluster_key: main_cluster
        - task_key: build_gold
          depends_on:
            - task_key: transform_silver
          notebook_task:
            notebook_path: ./notebooks/03_serve
          job_cluster_key: main_cluster
```